# Attention Heatmaps — sCT-RDT Interpretability

This notebook introspects a **trained** (or randomly initialized) `Full_sCTRDT_Model` by:
1. Tracing a single synthetic or real light curve through the custom `sCTRDT_Attention` block.
2. Extracting the raw softmax attention matrix $A \in \mathbb{R}^{S \times S}$ (per head, first layer).
3. Plotting multi-head heatmaps with Seaborn to visually verify:
   - **PEG**: noisy timestep pairs have suppressed attention weights.
   - **LTDK**: attention is concentrated around temporally proximal observations (Supernovae bursts).

Run cells top to bottom. Works with **synthetic data** if `best_model.pth` / CSV are missing.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys, warnings
sys.path.insert(0, '..')   # make 'src' importable when running from notebooks/
warnings.filterwarnings('ignore')

import yaml
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from src.models.sct_rdt import Full_sCTRDT_Model
from src.data_engine.dataset import AstroDataset

print('Imports OK')

In [ ]:
# ── Config + Model ─────────────────────────────────────────────────────────────
CONFIG_PATH = '../config.yaml'
CKPT_PATH   = '../best_model.pth'
DEVICE      = torch.device('cpu')   # CPU is fine for single-sample viz

with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

model = Full_sCTRDT_Model(config['model']).to(DEVICE)

try:
    # FAILSAFE: weights_only=True avoids arbitrary code execution warnings (PyTorch ≥ 2.0)
    model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE, weights_only=True))
    print(f'Loaded checkpoint: {CKPT_PATH}')
except FileNotFoundError:
    print(f'[WARN] {CKPT_PATH} not found — using randomly initialised weights.')

model.eval()
print('Model ready.')

In [ ]:
# ── Pick a single sample (synthetic) ──────────────────────────────────────────
MAX_SEQ_LEN = config['data']['max_seq_len']
SHOW_LEN    = 64    # Number of timesteps to visualise (keep heatmap readable)

ds = AstroDataset('../' + config['data']['train_path'], MAX_SEQ_LEN, occlusion_level='0%')
flux, passband, t_raw, err, mask, label = ds[0]

# Add batch dimension
flux     = flux.unsqueeze(0).to(DEVICE)       # [1, S]
passband = passband.unsqueeze(0).to(DEVICE)   # [1, S]
t_raw    = t_raw.unsqueeze(0).to(DEVICE)      # [1, S]
err      = err.unsqueeze(0).to(DEVICE)        # [1, S]
pad_mask = mask.unsqueeze(0).unsqueeze(1).unsqueeze(1).to(DEVICE)  # [1,1,1,S]

print(f'Sample | label={label.item()} | real timesteps={(~mask).sum().item()} / {MAX_SEQ_LEN}')

In [ ]:
# ── Hook: extract raw attention matrix from the FIRST encoder block ────────────
# We patch sCTRDT_Attention.forward to capture the softmax A matrix.
import torch.nn.functional as F
import math

_captured_A = {}

def _attn_hook(module, args, output):
    """Forward hook on the first sCTRDT_Attention block."""
    # Re-run the forward logic in no-grad mode just to get A
    # (output is already the projected result; we need the softmax internals)
    pass   # The hook below intercepts before W_out

# Instead, monkey-patch the attention forward to also store A
orig_forward = model.layers[0].attn.forward.__func__

def patched_forward(self, x, t_raw_in, error, pad_mask_in=None):
    B, S, _ = x.shape
    q = self.W_q(x).view(B, S, self.num_heads, self.d_k)
    k = self.W_k(x).view(B, S, self.num_heads, self.d_k)
    v = self.W_v(x).view(B, S, self.num_heads, self.d_k)

    q_rot, k_rot, t_scaled = self.scRoPE(q, k, t_raw_in)
    q_rot = q_rot.transpose(1, 2)
    k_rot = k_rot.transpose(1, 2)
    v     = v.transpose(1, 2)

    S_base = torch.matmul(q_rot, k_rot.transpose(-2, -1)) / math.sqrt(self.d_k)

    time_gaps  = torch.abs(t_scaled.unsqueeze(2) - t_scaled.unsqueeze(1)).unsqueeze(1)
    lambda_hat = F.softplus(self.lambda_h).view(1, self.num_heads, 1, 1)
    K_decay    = -(lambda_hat * time_gaps)

    err_i       = error.unsqueeze(2).expand(-1, -1, S).unsqueeze(-1)
    err_j       = error.unsqueeze(1).expand(-1, S, -1).unsqueeze(-1)
    error_pairs = torch.cat([err_i, err_j], dim=-1)
    gate_logits = self.W_g(error_pairs).permute(0, 3, 1, 2)
    K_noise     = torch.log(torch.sigmoid(gate_logits) + self.gamma)

    S_mat = S_base + K_decay + K_noise
    if pad_mask_in is not None:
        S_mat = S_mat.masked_fill(pad_mask_in, float('-inf'))

    A = F.softmax(S_mat, dim=-1)
    A = torch.nan_to_num(A, nan=0.0)

    # ── STORE for visualisation ──
    _captured_A['A'] = A.detach().cpu()

    out = torch.matmul(A, v).transpose(1, 2).contiguous().view(B, S, -1)
    return self.W_out(out)

import types
model.layers[0].attn.forward = types.MethodType(patched_forward, model.layers[0].attn)

# Forward pass
with torch.no_grad():
    logits = model(flux, passband, t_raw, err, pad_mask)

A_all = _captured_A['A']   # shape: [1, num_heads, S, S]
print(f'Captured A shape: {A_all.shape}')

In [ ]:
# ── Plot: multi-head attention heatmaps ───────────────────────────────────────
NUM_HEADS = config['model']['num_heads']
ncols     = 4
nrows     = (NUM_HEADS + ncols - 1) // ncols

A_vis = A_all[0, :, :SHOW_LEN, :SHOW_LEN].numpy()   # [num_heads, SL, SL]

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows))
axes = axes.flatten()

for h in range(NUM_HEADS):
    ax = axes[h]
    sns.heatmap(
        A_vis[h],
        ax=ax,
        cmap='viridis',
        vmin=0, vmax=A_vis[h].max(),
        cbar=True,
        square=True,
        xticklabels=False,
        yticklabels=False
    )
    ax.set_title(f'Head {h}', fontsize=10)
    ax.set_xlabel('Key timestep')
    ax.set_ylabel('Query timestep')

# Hide unused subplots
for ax in axes[NUM_HEADS:]:
    ax.set_visible(False)

fig.suptitle(
    f'Layer 0 — Softmax Attention $A_{{ij}}$ (first {SHOW_LEN} timesteps)\n'
    'Brighter = higher attention weight',
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.savefig('attention_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved attention_heatmaps.png')

In [ ]:
# ── Sanity checks ─────────────────────────────────────────────────────────────
# 1) Row sums should be ≈ 1 for real (non-padding) rows
row_sums = A_all[0].sum(dim=-1)   # [num_heads, S]
real_mask = ~mask[:SHOW_LEN]      # True where real data

print(f'Row-sum stats over real timesteps (expect ≈ 1.0):')
for h in range(NUM_HEADS):
    rs = row_sums[h, :SHOW_LEN][real_mask]
    if len(rs) > 0:
        print(f'  Head {h}: mean={rs.mean():.4f}  std={rs.std():.4f}  min={rs.min():.4f}  max={rs.max():.4f}')

# 2) Check for NaN — should be zero after our nan_to_num failsafe
nan_count = A_all.isnan().sum().item()
print(f'\nNaN count in A: {nan_count}  (should be 0)')

# 3) Classification result
_, pred_class = torch.max(logits, 1)
print(f'\nPredicted class: {pred_class.item()}  |  True label: {label.item()}')